# Qwen2.5 Kaggle Fine-tuning Notebook

Notebook này fine-tune `unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit` bằng Unsloth + QLoRA theo spec.

Run order cho người mới:
1. Chạy cell setup/import.
2. Giữ `RUN_MODE = "smoke"` để kiểm tra dữ liệu, format, token length, và train nhỏ.
3. Chỉ đổi sang `RUN_MODE = "full"` khi smoke run ổn.
4. Xem `mix_report.json`, `token_length_report.json`, và `eval_report.json` trước khi push adapter.

Notebook ưu tiên code đơn giản, từng bước rõ ràng, dễ sửa và dễ debug.

In [ ]:
!pip install unsloth unsloth-zoo rouge_score -q

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

In [ ]:
import json
import os
import random
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import math
import torch
import unsloth
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, TrainingArguments

# Unsloth / TRL imports are kept here so the notebook reads top-to-bottom.
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer

try:
    import wandb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb"])
    import wandb

# MODEL_ID = "unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit"
MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit"

DATASET_REPO = "tontide1/Dataset-for-fine-tuning-LLMS-VLSP-2023-benchmark"
DATASET_REVISION = "main"
SEED = 3407
MAX_SEQ_LENGTH = 2048
RUN_MODE = "full"  # đổi sang "full" khi smoke run ổn

SMOKE_CAPS: dict[str, int] = {
    "comprehension_short_answer": 100,
    "exams_mcq": 100,
    "wiki_mcq": 100,
    "instruction_retention": 100,
    "cloze_lm_retention": 100,
}

FULL_CAPS: dict[str, int] = {
    "comprehension_short_answer": 14_000,
    "exams_mcq": 10_760,
    "wiki_mcq": 7_136,
    "instruction_retention": 8_000,
    "cloze_lm_retention": 4_000,
}

if Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path.cwd()

ARTIFACT_DIR = BASE_DIR / "qwen25_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = ARTIFACT_DIR / "hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = ARTIFACT_DIR / "lora_model"
REPORT_DIR = ARTIFACT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL_NAME = "train_all.jsonl"
VAL_JSONL_NAME = "val_all.jsonl"
SHADOW_JSONL_NAME = "shadow_eval.jsonl"
INSTRUCTION_PROBE_JSONL_NAME = "instruction_probe.jsonl"
CLOZE_PROBE_JSONL_NAME = "cloze_probe.jsonl"
SPLIT_REPORT_NAME = "split_report.json"

TOKEN_STAT_KEYS = {
    "token_length",
    "total_tokens",
    "prompt_tokens",
    "completion_tokens",
    "input_tokens",
    "output_tokens",
    "n_tokens",
}


def get_kaggle_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        return None


HF_TOKEN = os.getenv("HF_TOKEN") or get_kaggle_secret("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY") or get_kaggle_secret("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

WANDB_PROJECT = "qwen25-vlsp-finetune"
WANDB_RUN_NAME = f"qwen25-{RUN_MODE}-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
WANDB_TAGS = [RUN_MODE, "qwen2.5-3b", "qlora", "vlsp"]
WANDB_RUN = None

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("MODEL_ID:", MODEL_ID)
print("DATASET_REPO:", DATASET_REPO)
print("DATASET_REVISION:", DATASET_REVISION)
print("RUN_MODE:", RUN_MODE)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("HF_TOKEN available:", HF_TOKEN is not None)
print("WANDB_API_KEY available:", WANDB_API_KEY is not None)
print("WANDB_PROJECT:", WANDB_PROJECT)
print("WANDB_RUN_NAME:", WANDB_RUN_NAME)


def init_wandb_run(mix_report: dict[str, Any], length_report: dict[str, Any]) -> Any | None:
    if not WANDB_API_KEY:
        print("W&B disabled: missing WANDB_API_KEY in env or Kaggle Secrets.")
        return None

    wandb.login(key=WANDB_API_KEY, relogin=True)
    run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        tags=WANDB_TAGS,
        dir=str(ARTIFACT_DIR),
        config={
            "model_id": MODEL_ID,
            "dataset_repo": DATASET_REPO,
            "dataset_revision": DATASET_REVISION,
            "seed": SEED,
            "run_mode": RUN_MODE,
            "max_seq_length": MAX_SEQ_LENGTH,
            "caps": mix_report["caps"],
            "train_counts": mix_report["train_counts"],
            "val_counts": mix_report["val_counts"],
            "length_summary_train": length_report["summary_train"],
            "length_summary_val": length_report["summary_val"],
        },
    )
    print("W&B run:", run.url)
    return run


def log_wandb_json(path: Path, artifact_type: str) -> None:
    if WANDB_RUN is None or not path.exists():
        return
    artifact = wandb.Artifact(path.stem, type=artifact_type)
    artifact.add_file(str(path))
    WANDB_RUN.log_artifact(artifact)

In [ ]:
def download_jsonl(filename: str) -> Path:
    """Download one file from the Hugging Face dataset repo."""
    path_str = hf_hub_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        revision=DATASET_REVISION,
        filename=filename,
        cache_dir=str(CACHE_DIR),
        token=HF_TOKEN,
    )
    return Path(path_str)
import wandb
print(wandb.__version__)


def read_jsonl(path: Path, max_lines: int | None = None) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if max_lines is not None and idx >= max_lines:
                break
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_row(row: dict[str, Any]) -> dict[str, Any]:
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) != 2:
        raise ValueError("Each row must contain exactly 2 messages: user and assistant.")

    metadata = row.get("metadata") or {}
    if not isinstance(metadata, dict):
        metadata = {}

    normalized: dict[str, Any] = {
        "messages": messages,
        "task": str(metadata.get("task", "unknown")),
        "subject": str(metadata.get("subject", "")),
    }

    for key in TOKEN_STAT_KEYS:
        if key in row:
            normalized[key] = row[key]
        elif key in metadata:
            normalized[key] = metadata[key]

    return normalized


def group_by_task(rows: list[dict[str, Any]]) -> dict[str, list[dict[str, Any]]]:
    buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        buckets[str(row.get("task", "unknown"))].append(row)
    return buckets


def make_training_mix(rows: list[dict[str, Any]], caps: dict[str, int], seed: int) -> list[dict[str, Any]]:
    rng = random.Random(seed)
    buckets = group_by_task(rows)

    mixed: list[dict[str, Any]] = []
    for task in sorted(caps):
        task_rows = list(buckets.get(task, []))
        rng.shuffle(task_rows)
        mixed.extend(task_rows[: min(len(task_rows), caps[task])])

    rng.shuffle(mixed)
    return mixed


def save_json(path: Path, data: dict[str, Any]) -> None:
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def count_tasks(rows: list[dict[str, Any]]) -> dict[str, int]:
    return dict(Counter(str(row.get("task", "unknown")) for row in rows))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def messages_to_text(messages: list[dict[str, Any]]) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    eos = tokenizer.eos_token or ""
    if eos and not text.rstrip().endswith(eos):
        text = text + eos
    return text


def rows_to_dataset(rows: list[dict[str, Any]]) -> Dataset:
    texts: list[str] = []
    tasks: list[str] = []
    for row in rows:
        texts.append(messages_to_text(row["messages"]))
        tasks.append(str(row.get("task", "unknown")))
    return Dataset.from_list([{ "text": text, "task": task } for text, task in zip(texts, tasks)])


def token_length(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def percentile(values: list[int], pct: float) -> int:
    if not values:
        return 0
    values = sorted(values)
    index = int(round((len(values) - 1) * pct))
    return values[max(0, min(index, len(values) - 1))]


def length_summary(rows: list[dict[str, Any]], threshold: int = MAX_SEQ_LENGTH) -> dict[str, Any]:
    tasks = group_by_task(rows)
    summary: dict[str, Any] = {}
    for task in sorted(tasks):
        lengths = [token_length(messages_to_text(row["messages"])) for row in tasks[task]]
        if not lengths:
            continue
        summary[task] = {
            "n": len(lengths),
            "p50": percentile(lengths, 0.50),
            "p90": percentile(lengths, 0.90),
            "p95": percentile(lengths, 0.95),
            "p99": percentile(lengths, 0.99),
            "max": max(lengths),
            f"trunc>{threshold}": round(sum(length > threshold for length in lengths) / len(lengths), 6),
        }
    return summary


def write_length_report(train_rows: list[dict[str, Any]], val_rows: list[dict[str, Any]]) -> dict[str, Any]:
    report = {
        "generated_at": "manual-run",
        "model_id": MODEL_ID,
        "seed": SEED,
        "thresholds": [MAX_SEQ_LENGTH],
        "summary_train": length_summary(train_rows),
        "summary_val": length_summary(val_rows),
    }
    save_json(REPORT_DIR / "token_length_report.json", report)
    return report


def save_long_examples(rows: list[dict[str, Any]], threshold: int = MAX_SEQ_LENGTH) -> Path:
    out_path = REPORT_DIR / f"long_examples_over_{threshold}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for row in rows:
            text = messages_to_text(row["messages"])
            if token_length(text) > threshold:
                f.write(json.dumps({"task": row.get("task"), "messages": row["messages"]}, ensure_ascii=False) + "\n")
    return out_path


def print_formatted_sample(rows: list[dict[str, Any]]) -> None:
    if not rows:
        print("No rows to sample.")
        return
    sample = rows[0]
    sample_text = messages_to_text(sample["messages"])
    print("Task:", sample.get("task", "unknown"))
    print("Has assistant marker:", "<|im_start|>assistant\n" in sample_text)
    print(sample_text[:1200])


def split_eval_by_task(rows: list[dict[str, Any]]) -> dict[str, Dataset]:
    grouped = group_by_task(rows)
    eval_sets: dict[str, Dataset] = {}
    for task in sorted(grouped):
        eval_sets[task] = rows_to_dataset(grouped[task])
    return eval_sets

In [ ]:
train_path = download_jsonl(TRAIN_JSONL_NAME)
val_path = download_jsonl(VAL_JSONL_NAME)
shadow_path = download_jsonl(SHADOW_JSONL_NAME)
instruction_probe_path = download_jsonl(INSTRUCTION_PROBE_JSONL_NAME)
cloze_probe_path = download_jsonl(CLOZE_PROBE_JSONL_NAME)
split_report_path = download_jsonl(SPLIT_REPORT_NAME)

raw_train_rows = read_jsonl(train_path)
raw_val_rows = read_jsonl(val_path)
raw_shadow_rows = read_jsonl(shadow_path)
raw_instruction_probe_rows = read_jsonl(instruction_probe_path)
raw_cloze_probe_rows = read_jsonl(cloze_probe_path)
split_report = json.loads(split_report_path.read_text(encoding="utf-8"))

train_rows = [normalize_row(row) for row in raw_train_rows]
val_rows = [normalize_row(row) for row in raw_val_rows]
shadow_rows = [normalize_row(row) for row in raw_shadow_rows]
instruction_probe_rows = [normalize_row(row) for row in raw_instruction_probe_rows]
cloze_probe_rows = [normalize_row(row) for row in raw_cloze_probe_rows]

print("Train rows:", len(train_rows))
print("Val rows:", len(val_rows))
print("Shadow rows:", len(shadow_rows))
print("Instruction probe rows:", len(instruction_probe_rows))
print("Cloze probe rows:", len(cloze_probe_rows))
print("Train task counts:", count_tasks(train_rows))
print("Val task counts:", count_tasks(val_rows))
print("Split report keys:", sorted(split_report.keys()))

current_caps = FULL_CAPS if RUN_MODE == "full" else SMOKE_CAPS
train_mix_rows = make_training_mix(train_rows, current_caps, SEED)
train_mix_counts = count_tasks(train_mix_rows)
val_counts = count_tasks(val_rows)

mix_report = {
    "model_id": MODEL_ID,
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "seed": SEED,
    "run_mode": RUN_MODE,
    "caps": current_caps,
    "train_counts": train_mix_counts,
    "val_counts": val_counts,
    "source_files": {
        "train": str(train_path),
        "val": str(val_path),
        "shadow": str(shadow_path),
        "instruction_probe": str(instruction_probe_path),
        "cloze_probe": str(cloze_probe_path),
        "split_report": str(split_report_path),
    },
}
save_json(REPORT_DIR / "mix_report.json", mix_report)

print("Mix counts:", train_mix_counts)
print("Current caps:", current_caps)

length_report = write_length_report(train_mix_rows, val_rows)
long_examples_path = save_long_examples(train_mix_rows)
print("Saved length report to:", REPORT_DIR / "token_length_report.json")
print("Saved long examples to:", long_examples_path)
print("Train length summary:", length_report["summary_train"].keys())
print("Val length summary:", length_report["summary_val"].keys())

WANDB_RUN = init_wandb_run(mix_report, length_report)
log_wandb_json(REPORT_DIR / "mix_report.json", "mix-report")
log_wandb_json(REPORT_DIR / "token_length_report.json", "token-length-report")

print_formatted_sample(train_mix_rows)

In [ ]:
from rouge_score import rouge_scorer
import re
from typing import Literal

def parse_mcq_answer(text: str) -> str | None:
    text = text.strip()
    m = re.search(r'\b([A-D])\b', text)
    if m:
        return m.group(1)
    if text in ('A', 'B', 'C', 'D'):
        return text
    return None

def batch_generate(
    rows: list[dict[str, Any]],
    model: Any,
    tokenizer: Any,
    max_new_tokens: int = 128,
    batch_size: int = 1,
) -> list[str]:
    outputs: list[str] = []
    for i in range(0, len(rows), batch_size):
        batch_rows = rows[i : i + batch_size]
        texts: list[str] = []
        for row in batch_rows:
            prompt_text = tokenizer.apply_chat_template(
                [row["messages"][0]],
                tokenize=False,
                add_generation_prompt=True,
            )
            texts.append(prompt_text)
        device = next(model.parameters()).device
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            decoded = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                do_sample=True,
                temperature=0.2,
                top_p=0.9,
            )
        for ids, text in zip(decoded, tokenizer.batch_decode(decoded, skip_special_tokens=True)):
            outputs.append(text)
        if (i // batch_size + 1) % 50 == 0:
            print(f"  Generated {min(i + batch_size, len(rows))}/{len(rows)}")
    return outputs

def compute_mcq_metrics(
    rows: list[dict[str, Any]],
    outputs: list[str],
    task: str,
) -> dict[str, Any]:
    correct = 0
    correct_norm = 0
    invalid = 0
    subject_correct: dict[str, int] = {}
    subject_total: dict[str, int] = {}
    for row, output in zip(rows, outputs):
        label = row["messages"][1]["content"].strip()
        pred = parse_mcq_answer(output)
        subject = str(row.get("subject", ""))
        if not subject:
            subject = "unknown"
        subject_total[subject] = subject_total.get(subject, 0) + 1
        if pred is None:
            invalid += 1
        else:
            if pred == label:
                correct += 1
                correct_norm += 1
                subject_correct[subject] = subject_correct.get(subject, 0) + 1
        if pred is not None and pred != label:
            correct_norm += 0
    total = len(rows)
    invalid_rate = invalid / total if total else 0
    accuracy = correct / total if total else 0
    acc_norm = correct_norm / (total - invalid) if (total - invalid) else 0
    result: dict[str, Any] = {
        f"{task}_accuracy": accuracy,
        f"{task}_acc_norm": acc_norm,
        f"{task}_invalid_answer_rate": invalid_rate,
        f"{task}_invalid_count": invalid,
        f"{task}_total": total,
    }
    if task == "exams_mcq" and subject_total:
        for subj in sorted(subject_total):
            s_acc = subject_correct.get(subj, 0) / subject_total[subj]
            result[f"{task}_macro_accuracy_by_subject/{subj}"] = s_acc
        result[f"{task}_macro_accuracy_by_subject/_mean"] = sum(
            subject_correct.get(s, 0) / subject_total[s] for s in subject_total
        ) / len(subject_total)
    return result

def compute_short_answer_metrics(
    rows: list[dict[str, Any]],
    outputs: list[str],
) -> dict[str, Any]:
    from collections import Counter
    empty = 0
    exact_matches = 0
    token_f1_scores: list[float] = []
    rouge_l_scores: list[float] = []
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    for row, output in zip(rows, outputs):
        answer = row["messages"][1]["content"].strip()
        pred = output.strip()
        if not pred or pred.lower() in ("null", "none", "n/a", "-"):
            empty += 1
        if pred.lower() == answer.lower():
            exact_matches += 1
        pred_toks = set(pred.split())
        answer_toks = set(answer.split())
        if pred_toks or answer_toks:
            tp = len(pred_toks & answer_toks)
            fp = len(pred_toks - answer_toks)
            fn = len(answer_toks - pred_toks)
            prec = tp / (tp + fp) if (tp + fp) else 0
            rec = tp / (tp + fn) if (tp + fn) else 0
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
            token_f1_scores.append(f1)
        else:
            token_f1_scores.append(0.0)
        rouge_result = scorer.score(answer, pred)
        rouge_l_scores.append(rouge_result["rougeL"].fmeasure)
    total = len(rows)
    return {
        "comprehension_short_answer_exact_match": exact_matches / total if total else 0,
        "comprehension_short_answer_token_f1": sum(token_f1_scores) / len(token_f1_scores) if token_f1_scores else 0,
        "comprehension_short_answer_rouge_l": sum(rouge_l_scores) / len(rouge_l_scores) if rouge_l_scores else 0,
        "comprehension_short_answer_empty_answer_rate": empty / total if total else 0,
        "comprehension_short_answer_empty_count": empty,
        "comprehension_short_answer_total": total,
    }

def compute_retention_metrics(
    rows: list[dict[str, Any]],
    outputs: list[str],
) -> dict[str, Any]:
    return {}

print("Metric functions loaded.")


In [ ]:
def choose_dtype() -> torch.dtype | None:
    if not torch.cuda.is_available():
        return None
    gpu_name = torch.cuda.get_device_name(0)
    if "T4" in gpu_name:
        return torch.float16
    return None


def build_data_collator(tokenizer: Any) -> tuple[Any | None, str]:
    response_template = "<|im_start|>assistant\n"
    try:
        try:
            from trl import DataCollatorForCompletionOnlyLM
        except Exception:
            from trl.trainer.utils import DataCollatorForCompletionOnlyLM

        collator = DataCollatorForCompletionOnlyLM(
            response_template=response_template,
            tokenizer=tokenizer,
        )
        return collator, "assistant_only_completion"
    except Exception:
        return None, "full_sequence_fallback"


model_dtype = choose_dtype()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=model_dtype,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

DATA_COLLATOR, loss_mode = build_data_collator(tokenizer)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("model_dtype:", model_dtype)
print("loss_mode:", loss_mode)
print("trainable parameters:")
model.print_trainable_parameters()


def build_trainer(train_dataset: Dataset, eval_dataset: Dataset) -> SFTTrainer:
    smoke_run = RUN_MODE == "smoke"
    args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_ratio=0.03 if not smoke_run else 0.0,
        warmup_steps=5 if smoke_run else 0,
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=torch.cuda.is_available() and not is_bfloat16_supported(),
        bf16=bool(torch.cuda.is_available() and is_bfloat16_supported()),
        logging_steps=1 if smoke_run else 10,
        save_steps=10 if smoke_run else 250,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=10 if smoke_run else 250,
        save_strategy="steps",
        max_steps=20 if smoke_run else -1,
        num_train_epochs=1 if not smoke_run else 1,
        report_to="wandb" if WANDB_RUN is not None else "none",
        seed=SEED,
        remove_unused_columns=False,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        average_tokens_across_devices=False,
        disable_tqdm=True,
    )

    trainer_kwargs: dict[str, Any] = {
        "model": model,
        "tokenizer": tokenizer,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "dataset_text_field": "text",
        "max_seq_length": MAX_SEQ_LENGTH,
        "dataset_num_proc": 1,
        "packing": False,
        "args": args,
    }
    if DATA_COLLATOR is not None:
        trainer_kwargs["data_collator"] = DATA_COLLATOR

    return SFTTrainer(**trainer_kwargs)


def tokenize_eval_dataset(dataset: Dataset) -> Dataset:
    """Create input_ids for datasets evaluated after trainer initialization."""
    if "input_ids" in dataset.column_names:
        return dataset

    def tokenize_batch(batch: dict[str, list[str]]) -> dict[str, Any]:
        return tokenizer(
            batch["text"],
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        )

    return dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=list(dataset.column_names),
        desc="Tokenizing eval dataset",
    )


def prepare_eval_dataset(trainer: SFTTrainer, dataset: Dataset, name: str) -> Dataset:
    """Use TRL preparation when available, then fall back to simple tokenization."""
    if "input_ids" in dataset.column_names:
        return dataset

    try:
        processing_class = getattr(trainer, "processing_class", tokenizer)
        eval_packing = getattr(trainer.args, "eval_packing", None)
        packing = getattr(trainer.args, "packing", False) if eval_packing is None else eval_packing
        return trainer._prepare_dataset(dataset, processing_class, trainer.args, packing, None, name)
    except Exception as exc:
        print(f"Falling back to manual tokenization for {name}: {exc}")
        return tokenize_eval_dataset(dataset)


def run_eval_by_task(trainer: SFTTrainer, datasets_by_task: dict[str, Dataset]) -> dict[str, Any]:
    results: dict[str, Any] = {}
    for task, dataset in datasets_by_task.items():
        prepared_dataset = prepare_eval_dataset(trainer, dataset, f"eval_{task}")
        metrics = trainer.evaluate(eval_dataset=prepared_dataset, metric_key_prefix=f"eval_{task}")
        results[task] = metrics
    return results


def weighted_main_score(task_metrics: dict[str, Any]) -> float:
    losses: list[float] = []
    for task in ["exams_mcq", "wiki_mcq", "comprehension_short_answer"]:
        task_result = task_metrics.get(task, {})
        loss = task_result.get(f"eval_{task}_loss")
        if loss is not None:
            losses.append(float(loss))
    return sum(losses) / len(losses) if losses else float("inf")


def collect_probe_metrics(trainer: SFTTrainer) -> dict[str, Any]:
    probes = {
        "instruction_probe": rows_to_dataset(instruction_probe_rows),
        "cloze_probe": rows_to_dataset(cloze_probe_rows),
    }
    probe_results: dict[str, Any] = {}
    for name, dataset in probes.items():
        prepared_dataset = prepare_eval_dataset(trainer, dataset, name)
        probe_results[name] = trainer.evaluate(eval_dataset=prepared_dataset, metric_key_prefix=name)
    return probe_results


def remove_notebook_progress_callback(trainer: SFTTrainer) -> None:
    """Avoid Kaggle notebook callback errors during post-training evaluate calls."""
    callbacks = trainer.callback_handler.callbacks
    kept_callbacks = [cb for cb in callbacks if cb.__class__.__name__ != "NotebookProgressCallback"]
    removed_count = len(callbacks) - len(kept_callbacks)
    trainer.callback_handler.callbacks = kept_callbacks
    if removed_count:
        print(f"Removed {removed_count} NotebookProgressCallback before post-training evaluation.")

In [ ]:
train_dataset = rows_to_dataset(train_mix_rows)
val_dataset = rows_to_dataset(val_rows)
eval_by_task = split_eval_by_task(val_rows)
shadow_dataset = rows_to_dataset(shadow_rows)

trainer = build_trainer(train_dataset, val_dataset)
train_output = trainer.train()
remove_notebook_progress_callback(trainer)

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

aggregate_metrics = trainer.evaluate(metric_key_prefix="eval_all")
task_metrics = run_eval_by_task(trainer, eval_by_task)
prepared_shadow_dataset = prepare_eval_dataset(trainer, shadow_dataset, "shadow")
shadow_metrics = trainer.evaluate(eval_dataset=prepared_shadow_dataset, metric_key_prefix="shadow")
probe_metrics = collect_probe_metrics(trainer)
main_score = weighted_main_score(task_metrics)
selected_checkpoint_path = trainer.state.best_model_checkpoint or str(OUTPUT_DIR)
selection_rationale = "Trainer loaded the best checkpoint by eval_loss; main-task and probe metrics were computed after training."

FastLanguageModel.for_inference(trainer.model)

print("Selected checkpoint:", selected_checkpoint_path)
print("Main score:", main_score)
print("Selection rationale:", selection_rationale)

print("\n--- Post-training generation metrics (all val samples) ---")
all_val_rows: list[dict[str, Any]] = []
for task in ["exams_mcq", "wiki_mcq", "comprehension_short_answer"]:
    task_rows = [r for r in val_rows if r.get("task") == task]
    all_val_rows.extend(task_rows)
    print(f"Generating for {task}: {len(task_rows)} samples...")
    outputs = batch_generate(task_rows, trainer.model, tokenizer, max_new_tokens=128, batch_size=1)
    if task in ("exams_mcq", "wiki_mcq"):
        metrics = compute_mcq_metrics(task_rows, outputs, task)
    else:
        metrics = compute_short_answer_metrics(task_rows, outputs)
    for k, v in sorted(metrics.items()):
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

post_train_metrics = {}
for task in ["exams_mcq", "wiki_mcq", "comprehension_short_answer"]:
    task_rows = [r for r in val_rows if r.get("task") == task]
    outputs = batch_generate(task_rows, trainer.model, tokenizer, max_new_tokens=128, batch_size=1)
    if task in ("exams_mcq", "wiki_mcq"):
        if task in ("exams_mcq", "wiki_mcq"):
            raw = compute_mcq_metrics(task_rows, outputs, task)
            for k, v in raw.items():
                post_train_metrics[f"post_train/{k}"] = v
    else:
            raw = compute_short_answer_metrics(task_rows, outputs)
            for k, v in raw.items():
                post_train_metrics[f"post_train/{k}"] = v

if probe_metrics:
    for name, pm in probe_metrics.items():
        loss = pm.get(f"{name}_loss")
        if loss is not None:
            if name == "instruction_probe":
                post_train_metrics[f"retention_{name}_loss"] = loss
            elif name == "cloze_probe":
                post_train_metrics[f"retention_{name}_perplexity"] = math.exp(loss)

print(f"\nPost-train metrics: {post_train_metrics}")

sample_outputs: list[str] = []
for row in val_rows[:3]:
    task = row.get("task", "unknown")
    prompt_text = tokenizer.apply_chat_template(
        [row["messages"][0]],
        tokenize=False,
        add_generation_prompt=True,
    )
    device = next(trainer.model.parameters()).device
    inputs = tokenizer([prompt_text], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
        )
    sample_outputs.append(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

for idx, output in enumerate(sample_outputs, start=1):
    print(f"\n=== Sample {idx} ===")
    print(output[:1500])

run_report = {
    "model_id": MODEL_ID,
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "seed": SEED,
    "run_mode": RUN_MODE,
    "loss_mode": loss_mode,
    "selected_checkpoint_path": selected_checkpoint_path,
    "selection_rationale": selection_rationale,
    "main_score": main_score,
    "train_metrics": getattr(train_output, "metrics", {}),
    "aggregate_metrics": aggregate_metrics,
    "task_metrics": task_metrics,
    "shadow_metrics": shadow_metrics,
    "probe_metrics": probe_metrics,
    "post_train_metrics": post_train_metrics,
    "sample_outputs": sample_outputs,
}
eval_report_path = REPORT_DIR / "eval_report.json"
save_json(eval_report_path, run_report)

if WANDB_RUN is not None:
    wandb_log_dict: dict[str, Any] = {
        "main_score": main_score,
        "eval_all_loss": aggregate_metrics.get("eval_all_loss"),
        "shadow_loss": shadow_metrics.get("shadow_loss"),
    }
    for key, value in post_train_metrics.items():
        if isinstance(value, (int, float)):
            wandb_log_dict[key] = value
    wandb.log(wandb_log_dict)
    WANDB_RUN.summary["selected_checkpoint_path"] = selected_checkpoint_path
    WANDB_RUN.summary["selection_rationale"] = selection_rationale
    WANDB_RUN.summary["loss_mode"] = loss_mode
    WANDB_RUN.summary["main_score"] = main_score
    for key, value in post_train_metrics.items():
        if isinstance(value, (int, float)):
            WANDB_RUN.summary[key] = value
    log_wandb_json(eval_report_path, "eval-report")

PUSH_TO_HUB = False
HF_REPO_ID = "tontide1/Qwen2.5-1.5B-VLSP-Adapter"
if PUSH_TO_HUB and HF_TOKEN:
    trainer.model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
else:
    print("Push skipped. Set PUSH_TO_HUB = True and provide HF_TOKEN when you are ready to upload.")

if WANDB_RUN is not None:
    wandb.finish()

print("Done.")
